# Taxi IA Chapinero — Análisis Exploratorio de Datos (EDA)

**Fuente:** Velocidades Bitcarrier, mayo de 2022 (servicio ArcGIS, SDM Bogotá).  
**Alcance:** caracterización de 46 segmentos históricos, todavía sin correspondencia validada con OSM.  
**Objetivo:** evaluar disponibilidad, distribución, valores extremos, variación temporal y pertinencia de las variables para una futura regresión de tiempos de desplazamiento.

> **Importante:** el EDA utiliza solo datos históricos observados. Los 292.488 escenarios OSM generados previamente son simulaciones independientes; no se usan para calcular estadísticas históricas ni para evaluar la regresión. Las unidades de distancia y velocidad están por confirmar documentalmente. Los tiempos calculados se presentan únicamente como referencia **derivada condicional**.

## 1. Librerías, fuente y reproducibilidad

Puedes ejecutar el notebook localmente desde VS Code/Jupyter. Si falta `matplotlib`, instala: `python -m pip install matplotlib ipykernel`. El archivo de entrada es el CSV comprimido generado por `preparar_bitcarrier.py`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

candidatos = [
    Path.cwd() / 'backend/datos/ml/bitcarrier_mayo_2022_procesado/observaciones_limpias.csv.gz',
    Path.cwd() / 'datos/ml/bitcarrier_mayo_2022_procesado/observaciones_limpias.csv.gz',
    Path.cwd().parent / 'datos/ml/bitcarrier_mayo_2022_procesado/observaciones_limpias.csv.gz',
    Path.cwd().parent.parent / 'backend/datos/ml/bitcarrier_mayo_2022_procesado/observaciones_limpias.csv.gz',
]
archivo = next((p for p in candidatos if p.is_file()), None)
if archivo is None:
    raise FileNotFoundError('No se encontró observaciones_limpias.csv.gz. Ejecuta preparar_bitcarrier.py o revisa la carpeta de trabajo de Jupyter.')
print('Fuente:', archivo.resolve())
datos = pd.read_csv(archivo)
print('Dimensiones:', datos.shape)
datos.head()

## 2. Estructura, faltantes, duplicados y validación básica

Para no confundir integridad con validez, verificamos duplicados, tipos, valores nulos y rangos. La columna de velocidad media mixta fue reportada nula en la extracción, por lo que el análisis usa `velocidad_original`.

In [ ]:
display(datos.dtypes.to_frame('tipo'))
display(datos.isna().sum().to_frame('nulos'))
print('ObjectID duplicados:', datos['objectid'].duplicated().sum())
print('Pares (TID, INICIO) duplicados:', datos.duplicated(['tid','inicio_original']).sum())
print('Tramos TID:', datos['tid'].nunique())
print('Velocidad no positiva:', (datos['velocidad_original'] <= 0).sum())
print('Distancia no positiva:', (datos['distancia_original'] <= 0).sum())
display(datos[['velocidad_original','distancia_original','num_dispositivos']].describe(percentiles=[.05,.25,.5,.75,.95,.99]).T)

## 3. Periodo y coherencia temporal

`INICIO` tiene formato epoch milisegundos. Al interpretarlo como UTC, la hora coincide con `HORA` en la fuente, pero **esto no demuestra** que la hora física de medición en Bogotá fuera UTC. Se conserva la hora y día reportados por Bitcarrier.

In [ ]:
datos['fecha_epoch_utc'] = pd.to_datetime(datos['inicio_original'], unit='ms', utc=True)
print('Inicio UTC:',datos['fecha_epoch_utc'].min())
print('Fin UTC:',datos['fecha_epoch_utc'].max())
print('Intervalos distintos:', datos['fecha_epoch_utc'].nunique())
print('Días UTC distintos:', datos['fecha_epoch_utc'].dt.date.nunique())
print('Horas coincidentes:', (datos['fecha_epoch_utc'].dt.hour == datos['hora']).sum(), '/', len(datos))

cobertura = datos.groupby(datos['fecha_epoch_utc'].dt.date).size()
fig, ax = plt.subplots(figsize=(12,3.5))
ax.plot(range(len(cobertura)), cobertura.values, marker='o')
ax.set_xticks(range(0,len(cobertura),3), [str(x) for x in cobertura.index[::3]], rotation=35)
ax.set_title('Observaciones disponibles por día — mayo 2022')
ax.set_ylabel('Observaciones'); ax.grid(alpha=.25); plt.tight_layout(); plt.show()

## 4. Distribuciones y valores atípicos

Los valores extremos se visualizan y se cuentan. **No se eliminan automáticamente**: deben investigarse según la calidad de la medición y las unidades del proveedor.

In [ ]:
fig, axes=plt.subplots(1,3,figsize=(15,4))
axes[0].hist(datos['velocidad_original'],bins=55);axes[0].set_title('Velocidad original');axes[0].set_xlabel('Unidad reportada')
axes[1].boxplot(datos['velocidad_original'],vert=True);axes[1].set_title('Valores extremos de velocidad')
axes[2].hist(datos['num_dispositivos'],bins=45);axes[2].set_title('Número de dispositivos');axes[2].set_xlabel('Dispositivos')
for ax in axes:ax.grid(alpha=.15)
plt.tight_layout();plt.show()
q1,q3=datos['velocidad_original'].quantile([.25,.75]);lim=q3+1.5*(q3-q1)
print(f'Velocidad >100: {(datos.velocidad_original>100).sum():,} observaciones')
print(f'Velocidad > límite IQR ({lim:.2f}): {(datos.velocidad_original>lim).sum():,}')
print(f'Registros con un dispositivo: {(datos.num_dispositivos==1).sum():,}')

## 5. Perfil horario y semanal

Estos perfiles describen el mes completo y permiten observar variaciones históricas. Para predecir días posteriores, **no debemos usarlos como característica calculada a partir del periodo de prueba**.

In [ ]:
orden=['Lunes','Martes','Miercoles','Jueves','Viernes','Sabado','Domingo']
horario=datos.groupby('hora')['velocidad_original'].agg(['median','mean','count']).reindex(range(24))
semanal=datos.groupby('dia_semana_original')['velocidad_original'].median().reindex(orden)
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].plot(horario.index,horario['median'],label='Mediana',marker='o',markersize=3)
axes[0].plot(horario.index,horario['mean'],label='Media',alpha=.7)
axes[0].set_title('Velocidad por hora (fuente)');axes[0].set_xticks(range(0,24,2));axes[0].legend()
axes[1].bar(semanal.index,semanal.values);axes[1].set_title('Velocidad mediana por día');axes[1].tick_params(axis='x',rotation=35)
for ax in axes:ax.set_ylabel('Velocidad original');ax.grid(alpha=.2)
plt.tight_layout();plt.show()

In [ ]:
matriz=datos.pivot_table(index='dia_semana_original',columns='hora',values='velocidad_original',aggfunc='median').reindex(orden)
fig,ax=plt.subplots(figsize=(13,4))
im=ax.imshow(matriz.to_numpy(),aspect='auto')
ax.set_yticks(range(len(orden)),orden);ax.set_xticks(range(24),[str(i) for i in range(24)])
ax.set_title('Mapa de calor: velocidad mediana por día y hora')
fig.colorbar(im,ax=ax,label='Velocidad original');plt.tight_layout();plt.show()

## 6. Diferencias entre tramos y calidad de medición

Cada TID identifica un segmento Bitcarrier. **No equivale a una arista OSM** hasta verificar geometría, nombres y sentido. Comparar tramos ayuda a identificar heterogeneidad y cobertura.

In [ ]:
top=datos['tid'].value_counts().head(12).index.tolist()
sub=datos[datos['tid'].isin(top)]
fig,ax=plt.subplots(figsize=(13,5))
ax.boxplot([sub.loc[sub.tid==tid,'velocidad_original'].values for tid in top],tick_labels=[str(x) for x in top],showfliers=False)
ax.set_title('Velocidad por TID — 12 tramos más observados (sin outliers en la figura)')
ax.set_xlabel('TID');ax.set_ylabel('Velocidad original');ax.tick_params(axis='x',rotation=45)
plt.tight_layout();plt.show()
cob_tids=datos.groupby('tid').agg(observaciones=('objectid','size'),instantes=('inicio_original','nunique'),velocidad_mediana=('velocidad_original','median'), dispositivos_mediana=('num_dispositivos','median')).sort_values('observaciones')
display(cob_tids.head(10));print('Máximo de intervalos esperado por tramo: 2976')

## 7. Relación entre variables y tiempo derivado (hipótesis de unidades)

La siguiente transformación es **hipotética** hasta confirmar la documentación de `DISTANCE` (m) y `VEL_PROMEDIO` (km/h):

\[t_{min}=0.06\,\frac{d_{m}}{v_{km/h}}\]

Este tiempo es un **objetivo calculado**, no el tiempo medido directamente de un taxi. La velocidad observada no puede incluirse como predictor del mismo registro, porque determina matemáticamente el objetivo y produciría fuga de información.

In [ ]:
mascara=(datos['distancia_original']>0) & (datos['velocidad_original']>0)
datos['tiempo_derivado_min']=np.nan
datos.loc[mascara,'tiempo_derivado_min']=.06*datos.loc[mascara,'distancia_original']/datos.loc[mascara,'velocidad_original']
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].hist(datos['tiempo_derivado_min'].clip(upper=datos['tiempo_derivado_min'].quantile(.99)),bins=50)
axes[0].set_title('Tiempo derivado, truncado visualmente en p99');axes[0].set_xlabel('Minutos hipotéticos')
# Muestra fija reproducible para evitar sobrecargar el gráfico
muestra=datos.sample(n=min(5000,len(datos)),random_state=42)
axes[1].scatter(muestra['distancia_original'],muestra['tiempo_derivado_min'],s=4,alpha=.13)
axes[1].set_title('Distancia vs tiempo derivado');axes[1].set_xlabel('Distancia original');axes[1].set_ylabel('Minutos hipotéticos')
plt.tight_layout();plt.show()
print(datos[['distancia_original','velocidad_original','num_dispositivos','tiempo_derivado_min']].corr().round(3))

## 8. Hallazgos y decisiones metodológicas

**Conclusiones a completar tras ejecutar el notebook** (no escribir resultados antes de observar las figuras):

1. ¿La velocidad es asimétrica? ¿Qué parte de la distribución presenta extremos y cómo se investigarán?
2. ¿Existen patrones por hora y día? ¿Son consistentes entre tramos?
3. ¿Qué TID presentan menor cobertura o valores discordantes?
4. ¿Cuántas mediciones dependen de un solo dispositivo? ¿Conviene analizar sensibilidad a ese filtro?
5. ¿La hipótesis de metros/km/h resulta razonable? **Confirmar con la fuente original antes del entrenamiento definitivo.**

**Decisiones para regresión:**

- Separar datos cronológicamente (no usar muestras futuras para generar estadísticas de entrenamiento).
- Comenzar por regresión lineal múltiple y una referencia ingenua, evaluadas con MAE, RMSE y R².
- Usar solo entradas disponibles al predecir: distancia, día, hora, tipo de vía (si se valida), etc.
- Evitar `velocidad_original`, `tiempo_derivado_min` o agregados de todo mayo como variables predictoras del mismo intervalo.
- No evaluar predicciones con escenarios sintéticos generados a partir de los propios datos.
- Evaluar aparte la generalización entre TID y hacia aristas OSM no observadas.

**Limitación:** sin validación espacial y direccional, los 46 tramos no pueden atribuirse a las 1741 aristas. El dashboard describe solo Bitcarrier.

## 9. Exportación del dashboard

El mismo conjunto de datos alimenta un JSON compacto mediante `backend/scripts/ml/generar_resumen_eda.py`; el notebook y Nuxt comparten cálculos sin llevar 134.538 filas al despliegue. Ejecutar desde la raíz del repositorio:

```powershell
python .\backend\scripts\ml\generar_resumen_eda.py
```

El archivo generado en `frontend/public/datos/eda_bitcarrier.json` **sí** puede versionarse; los datos originales y procesados seguirán excluidos por `.gitignore`. El análisis exploratorio del conjunto completo es descriptivo; para entrenamiento, cualquier estadística de preparación se recalcula dentro del periodo de entrenamiento.